### Json Parsing And Processing

In [ ]:
import json
import os

os.makedirs("data/json_files", exist_ok=True)

In [ ]:
# Sample nested JSON data
json_data = {
    "company": "TechCorp",
    "employees": [
        {
            "id": 1,
            "name": "John Doe",
            "role": "Software Engineer",
            "skills": ["Python", "JavaScript", "React"],
            "projects": [
                {"name": "RAG System", "status": "In Progress"},
                {"name": "Data Pipeline", "status": "Completed"},
            ],
        },
        {
            "id": 2,
            "name": "Jane Smith",
            "role": "Data Scientist",
            "skills": ["Python", "Machine Learning", "SQL"],
            "projects": [
                {"name": "ML Model", "status": "In Progress"},
                {"name": "Analytics Dashboard", "status": "Planning"},
            ],
        },
    ],
    "departments": {
        "engineering": {"head": "Mike Johnson", "budget": 1000000, "team_size": 25},
        "data_science": {"head": "Sarah Williams", "budget": 750000, "team_size": 15},
    },
}

In [ ]:
with open("data/json_files/company_data.json", "w") as f:
    json.dump(json_data, f, indent=2)

In [ ]:
# Save JSON Lines format
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},
    {"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99},
]

with open("data/json_files/events.jsonl", "w") as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + "\n")

## Json Processing Stratergies

In [ ]:
from langchain_community.document_loaders import JSONLoader
import json

## MEthod1 : JsonLoader With jq_schema
print("1️⃣ JSONLoader - Extract specific fields")

# Extract employee information
employee_loader = JSONLoader(
    file_path="data/json_files/company_data.json",
    jq_schema=".employees[]",  # jq query to extract only employee array
    text_content=False,  # Get full JSON as objects
)
employee_docs = employee_loader.load()
print(f"Loaded {len(employee_docs)} employee documents")
print(f"First employee: {employee_docs[0].page_content[:200]}...")
print(employee_docs)

# Extract whole JSON
whole_loader = JSONLoader(
    file_path="data/json_files/company_data.json",
    jq_schema=".",  # jq query to extract the root object (whole JSON)
    text_content=False,  # Get full JSON object
)
whole_docs = whole_loader.load()
print(f"Whole JSON: {whole_docs[0].page_content}")


/Users/nkd/nkdFiles/codes/my/github/dwnishant/ultimateRAGBootcamp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1️⃣ JSONLoader - Extract specific fields
Loaded 2 employee documents
First employee: {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status"...
[Document(metadata={'source': '/Users/nkd/nkdFiles/codes/my/github/dwnishant/ultimateRAGBootcamp/0-DataIngestParsing/data/json_files/company_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'), Document(metadata={'source': '/Users/nkd/nkdFiles/codes/my/github/dwnishant/ultimateRAGBootcamp/0-DataIngestParsing/data/json_files/company_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Jane Smith", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name":

## Understanding jq query language

### Short jq Tutorial

jq is a lightweight command-line JSON processor that uses a simple syntax for querying and transforming JSON data. It's commonly used in tools like LangChain's `JSONLoader` via the `jq_schema` parameter to extract specific parts of JSON files into documents.

#### Basic Concepts
- **Selectors**: Start with `.` (root). Use `.key` for object keys, `.[index]` for array elements, or `.[]` to iterate over arrays.
- **Output**: By default, jq outputs JSON. In `JSONLoader`, `jq_schema` defines what to extract as the document content.
- **Chaining**: Combine selectors with pipes `|` for more complex queries.

#### Common Examples (Based on Your JSON Structure)
Assuming your `company_data.json` looks like this:
```json
{
  "company": "TechCorp",
  "employees": [
    {"id": 1, "name": "John Doe", "role": "Software Engineer"},
    {"id": 2, "name": "Jane Smith", "role": "Data Scientist"}
  ],
  "departments": {
    "engineering": {"head": "Mike Johnson"},
    "data_science": {"head": "Sarah Williams"}
  }
}
```

- **Get the entire JSON**: `.`  
  *Example*: `jq_schema="."` → Returns the whole object as one document.

- **Get a specific key**: `.company`  
  *Example*: `jq_schema=".company"` → Returns `"TechCorp"`.

- **Iterate over an array**: `.employees[]`  
  *Example*: `jq_schema=".employees[]"` → Creates one document per employee object.

- **Access nested data**: `.departments.engineering.head`  
  *Example*: `jq_schema=".departments.engineering.head"` → Returns `"Mike Johnson"`.

- **Get array elements by index**: `.employees[0]`  
  *Example*: `jq_schema=".employees[0]"` → Returns the first employee object.

- **Filter arrays**: `.employees[] | select(.role == "Software Engineer")`  
  *Example*: `jq_schema=".employees[] | select(.role == \"Software Engineer\")"` → Returns only matching employees.

- **Combine with pipes**: `.departments | to_entries[] | .value.head`  
  *Example*: `jq_schema=".departments | to_entries[] | .value.head"` → Iterates over department heads.

#### Tips for JSONLoader
- Use `text_content=False` to get structured JSON objects in `page_content`.
- Test queries in a terminal with `jq '.query' file.json` to verify output.
- For complex structures, consider custom Python processing (like your `process_json_intelligently` function) if jq gets too intricate.

For more, check the [official jq manual](https://stedolan.github.io/jq/manual/). If you have a specific query in mind, provide the JSON snippet and desired output!

## Custom JSON Processing with Python

In [ ]:
# Method 2: Custom JSON processing for complex structures
from typing import List
from langchain_core.documents import Document

print("\n2️⃣ Custom JSON Processing")


def process_json_intelligently(filepath: str) -> List[Document]:
    """Process JSON with intelligent flattening and context preservation"""
    with open(filepath, "r") as f:
        data = json.load(f)

    documents = []

    # Strategy 1: Create documents for each employee with full context
    for emp in data.get("employees", []):
        content = f"""Employee Profile:
        Name: {emp["name"]}
        Role: {emp["role"]}
        Skills: {", ".join(emp["skills"])}

        Projects:"""
        for proj in emp.get("projects", []):
            content += f"\n- {proj['name']} (Status: {proj['status']})"

        doc = Document(
            page_content=content,
            metadata={
                "source": filepath,
                "data_type": "employee_profile",
                "employee_id": emp["id"],
                "employee_name": emp["name"],
                "role": emp["role"],
            },
        )
        documents.append(doc)

    return documents


2️⃣ Custom JSON Processing


In [ ]:
process_json_intelligently("data/json_files/company_data.json")

[Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 1, 'employee_name': 'John Doe', 'role': 'Software Engineer'}, page_content='Employee Profile:\n        Name: John Doe\n        Role: Software Engineer\n        Skills: Python, JavaScript, React\n\n        Projects:\n- RAG System (Status: In Progress)\n- Data Pipeline (Status: Completed)'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 2, 'employee_name': 'Jane Smith', 'role': 'Data Scientist'}, page_content='Employee Profile:\n        Name: Jane Smith\n        Role: Data Scientist\n        Skills: Python, Machine Learning, SQL\n\n        Projects:\n- ML Model (Status: In Progress)\n- Analytics Dashboard (Status: Planning)')]